# Shopee Price Match Guarantee — fill-in baseline

Problem family: **Multimodal product matching**. Official metric: **Mean per-query set F1,
including self**.

**Learning objective:** Training-title TF-IDF + batched cosine threshold matching.

[Dataset and original task](https://www.kaggle.com/competitions/shopee-product-matching) ·
[Primary solution
evidence](https://www.kaggle.com/competitions/shopee-product-matching/writeups/upstage-making-ai-beneficial-1st-place-solution-fr)

The first-place write-up emphasizes matching rules over marginal encoder gains and normalizing
each modality before fusion. This lesson implements frozen local encoder extraction,
normalize-then-concatenate matching and a small threshold comparison. It omits ArcFace
training and iterative neighborhood blending.

**Watch for:** Hold out entire product groups. Matching is transductive gallery inference, not
fitting on test labels. Include each item itself.

**Standalone:** this file contains every project-specific helper it uses. Only installed
third-party libraries and your local data/assets are required; no repository checkout, source
imports, project installation or external bootstrap is used. Only the optional dataset cell
needs internet when enabled; all training works offline. Edit DATA_DIR and
output/storage settings in stage 1. Train media on local disk, then persist your output bundle.

This notebook is a learning exercise, not a claimed leaderboard reproduction. No trained
weights or real Kaggle data are bundled. Keep grading answers closed. Use only explicitly
permitted resources. Internet/prepared-code access is not assumed on contest day.

Fill only the TODO blocks. Every code cell is followed by a closed Reveal expected code
section. Attempt, run, then reveal; equivalent correct implementations are welcome. Record
hints needed. Setup/configuration is supplied.

After the lesson, close both notebooks and rebuild the essential route from memory. See the
repository's blank-notebook playbook for the final transfer test.




## 1. Bootstrap and configuration

Run this supplied setup cell; edit only configuration values. Later algorithm cells contain
TODO blocks. No weights are fetched. The next optional cell can acquire practice data;
otherwise stage local data before repeated reads.
Each candidate gets a new output directory.






In [ ]:
import json
import pickle
import random
import shutil
import sys
import time
import zipfile
from collections.abc import Callable, Iterable, Mapping, Sequence
from dataclasses import asdict, dataclass, field, is_dataclass
from pathlib import Path
from time import perf_counter
from types import SimpleNamespace
from typing import Any
from uuid import uuid4

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader


def _json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if is_dataclass(value):
        return asdict(value)
    if hasattr(value, "tolist"):
        return value.tolist()
    raise TypeError(f"Cannot serialize {type(value).__name__} as JSON")


def _write(value: Any, path: Path | str, kind: str) -> Path:
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f".{destination.name}.{uuid4().hex}.tmp")
    try:
        if kind == "json":
            temporary.write_text(
                json.dumps(
                    value, default=_json_default, indent=2, ensure_ascii=False, allow_nan=False
                ),
                encoding="utf-8",
            )
        elif kind == "pickle":
            with temporary.open("wb") as handle:
                pickle.dump(value, handle, protocol=pickle.HIGHEST_PROTOCOL)
        else:
            import torch

            torch.save(value, temporary)
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination


def save_json(value: Any, path: Path | str) -> Path:
    """Atomically save UTF-8 metadata, including paths, dataclasses, and NumPy values."""
    return _write(value, path, "json")


def save_pickle(value: Any, path: Path | str) -> Path:
    """Atomically save a fitted local pipeline with preprocessing and original label values."""
    return _write(value, path, "pickle")


def load_pickle(path: Path | str) -> Any:
    """Reload a trusted, locally produced pickle; never load untrusted downloaded pickle files."""
    with Path(path).open("rb") as handle:
        return pickle.load(handle)


def save_torch_state(value: Any, path: Path | str) -> Path:
    """Atomically save torch state without importing torch until this function is used."""
    return _write(value, path, "torch")


@dataclass
class CompetitionConfig:
    """Explicit notebook state; no installed project is required."""

    data_dir: Path
    output_dir: Path
    seed: int = 42
    num_workers: int = 0
    pretrained_allowed: bool = False
    time_budget_minutes: float = 60
    device: str = "auto"

    def prepare(self) -> None:
        self.device = (
            ("cuda" if torch.cuda.is_available() else "cpu")
            if self.device == "auto"
            else self.device
        )
        self.output_dir.mkdir(parents=True, exist_ok=True)
        random.seed(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.seed)

    def to_dict(self) -> dict:
        return {**asdict(self), "data_dir": str(self.data_dir), "output_dir": str(self.output_dir)}


class TimeBudget:
    """Leave delivery time available; a running batch/fit cannot be interrupted safely."""

    def __init__(self, minutes: float, reserve_minutes: float = 10) -> None:
        if not 0 <= reserve_minutes < minutes:
            raise ValueError("Require 0 <= reserve < total minutes")
        self.deadline = time.monotonic() + 60 * (minutes - reserve_minutes)

    @property
    def should_stop(self) -> bool:
        return time.monotonic() >= self.deadline

    def checkpoint(self, stage: str) -> None:
        if self.should_stop:
            raise TimeoutError(f"Delivery reserve reached before {stage}")


def runtime_paths(
    *, runtime_dir: Path | None = None, persistent_dir: Path | None = None
) -> SimpleNamespace:
    """Media stays on local disk; optional durable storage is an explicit destination."""
    root = (
        Path(runtime_dir)
        if runtime_dir is not None
        else (Path("/content/olp_runtime") if "google.colab" in sys.modules else Path.cwd())
    )
    data, output = root / "data", root / "outputs"
    data.mkdir(parents=True, exist_ok=True)
    output.mkdir(parents=True, exist_ok=True)
    return SimpleNamespace(
        data_dir=data,
        output_dir=output,
        persistent_dir=Path(persistent_dir) if persistent_dir is not None else None,
    )


def mount_google_drive() -> None:
    """Call only when an explicitly configured Colab path uses Drive."""
    from google.colab import drive

    drive.mount("/content/drive")


def stage_data(source: Path | str, destination: Path | str) -> Path:
    """Copy a folder or extract a ZIP once, then train from the local destination."""
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if source == destination or destination.is_relative_to(source):
        raise ValueError("Staging destination must be separate from its source")
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError("Reuse staged DATA_DIR and set DATA_ARCHIVE=None")
    destination.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        with zipfile.ZipFile(source) as archive:
            for member in archive.infolist():
                if not (destination / member.filename).resolve().is_relative_to(destination):
                    raise ValueError("Archive contains a path outside the destination")
            archive.extractall(destination)
    return destination


def sync_artifacts(source: Path | str, destination: Path | str | None) -> list[Path]:
    """Copy the entire candidate, including encoders, preserving separate run folders."""
    if destination is None:
        return []
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if (
        destination == source
        or destination.is_relative_to(source)
        or source.is_relative_to(destination)
    ):
        raise ValueError("Output and backup folders must be separate")
    copied = []
    for path in source.rglob("*"):
        if path.is_file() and not path.name.endswith(".tmp"):
            target = destination / path.relative_to(source)
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_name(f".{target.name}.{uuid4().hex}.tmp")
            try:
                shutil.copy2(path, temporary)
                temporary.replace(target)
            finally:
                temporary.unlink(missing_ok=True)
            copied.append(target)
    return copied


def gpu_report() -> dict:
    """Inspect the actual runtime before choosing image size, batch size or precision."""
    if not torch.cuda.is_available():
        return {"device": "cpu", "precision": "float32"}
    free, total = torch.cuda.mem_get_info()
    return {
        "device": torch.cuda.get_device_name(),
        "free_GiB": free / 2**30,
        "total_GiB": total / 2**30,
        "precision": "bf16" if torch.cuda.is_bf16_supported() else "fp16",
    }


def evaluate_metric(name: str, truth: Any, prediction: Any, **kwargs: Any) -> float:
    """Only the metrics used in this lesson; task-specific graders appear beside the model."""
    from sklearn.metrics import accuracy_score, roc_auc_score

    if name == "roc_auc":
        return float(roc_auc_score(truth, prediction, **kwargs))
    if name == "accuracy":
        return float(accuracy_score(truth, prediction))
    if name == "dice_mean_image":
        y, p = np.asarray(truth, bool), np.asarray(prediction, bool)
        if y.shape != p.shape or not y.size:
            raise ValueError("Need aligned nonempty masks")
        axes = tuple(range(1, y.ndim))
        denominator = y.sum(axis=axes) + p.sum(axis=axes)
        return float(
            np.divide(
                2 * (y & p).sum(axis=axes), denominator, out=np.ones(len(y)), where=denominator != 0
            ).mean()
        )
    raise ValueError(f"Unsupported lesson metric: {name}")


FAMILY, VARIANT = "07_matching", "baseline"
DATA_ARCHIVE, PERSISTENT_DIR = None, None
if "google.colab" in sys.modules and any(
    "/content/drive/" in str(value) for value in (DATA_ARCHIVE, PERSISTENT_DIR)
):
    mount_google_drive()
paths = runtime_paths(persistent_dir=PERSISTENT_DIR)
DATA_DIR = (
    paths.data_dir / "learning" / FAMILY
    if "google.colab" in sys.modules
    else Path.cwd() / "data" / "learning" / FAMILY
)
if DATA_ARCHIVE is not None:
    stage_data(DATA_ARCHIVE, DATA_DIR)
RUN_ID = f"{FAMILY}-{VARIANT}-{uuid4().hex[:8]}"
OUTPUT = paths.output_dir / "learning" / RUN_ID
DURABLE_OUTPUT = paths.persistent_dir / RUN_ID if paths.persistent_dir is not None else None
PRETRAINED_ALLOWED, WEIGHT_PATH = False, None
SEED, MAX_ROWS, IMAGE_SIZE, BATCH_SIZE, EPOCHS = 42, 6000, 128, 16, 3
TIME_BUDGET_MINUTES, DELIVERY_RESERVE_MINUTES = 60, 10
GROUP_COLUMN, TIME_COLUMN = "label_group", None
USE_IMBALANCE_WEIGHTS = False
USE_HISTORY_FEATURES = False
MODEL_NAME = "resnet18"
IMAGE_SUFFIX = ".png"
TEXT_MODEL_DIR, IMAGE_MODEL_CHECKPOINT, EMBEDDING_ARCHIVE = None, None, None
ASSET_PROVENANCE = (
    ""  # exact asset/version/license; required when enabling local pretrained features
)
EMBEDDING_PATH, EMBEDDING_DIM, MAX_LENGTH = None, 64, 128
THRESHOLD_GRID = (0.3, 0.4, 0.5, 0.6, 0.7)
competition = CompetitionConfig(
    data_dir=DATA_DIR,
    output_dir=OUTPUT,
    seed=SEED,
    num_workers=0,
    pretrained_allowed=PRETRAINED_ALLOWED,
    time_budget_minutes=TIME_BUDGET_MINUTES,
)
competition.prepare()
budget = TimeBudget(TIME_BUDGET_MINUTES, reserve_minutes=DELIVERY_RESERVE_MINUTES)
print(gpu_report())
print("Local data:", DATA_DIR, "New candidate:", OUTPUT)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>import json
import pickle
import random
import shutil
import sys
import time
import zipfile
from collections.abc import Callable, Iterable, Mapping, Sequence
from dataclasses import asdict, dataclass, field, is_dataclass
from pathlib import Path
from time import perf_counter
from types import SimpleNamespace
from typing import Any
from uuid import uuid4

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader


def _json_default(value: Any) -&gt; Any:
    if isinstance(value, Path):
        return str(value)
    if is_dataclass(value):
        return asdict(value)
    if hasattr(value, &quot;tolist&quot;):
        return value.tolist()
    raise TypeError(f&quot;Cannot serialize {type(value).__name__} as JSON&quot;)


def _write(value: Any, path: Path | str, kind: str) -&gt; Path:
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f&quot;.{destination.name}.{uuid4().hex}.tmp&quot;)
    try:
        if kind == &quot;json&quot;:
            temporary.write_text(
                json.dumps(
                    value, default=_json_default, indent=2, ensure_ascii=False, allow_nan=False
                ),
                encoding=&quot;utf-8&quot;,
            )
        elif kind == &quot;pickle&quot;:
            with temporary.open(&quot;wb&quot;) as handle:
                pickle.dump(value, handle, protocol=pickle.HIGHEST_PROTOCOL)
        else:
            import torch

            torch.save(value, temporary)
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination


def save_json(value: Any, path: Path | str) -&gt; Path:
    &quot;&quot;&quot;Atomically save UTF-8 metadata, including paths, dataclasses, and NumPy values.&quot;&quot;&quot;
    return _write(value, path, &quot;json&quot;)


def save_pickle(value: Any, path: Path | str) -&gt; Path:
    &quot;&quot;&quot;Atomically save a fitted local pipeline with preprocessing and original label values.&quot;&quot;&quot;
    return _write(value, path, &quot;pickle&quot;)


def load_pickle(path: Path | str) -&gt; Any:
    &quot;&quot;&quot;Reload a trusted, locally produced pickle; never load untrusted downloaded pickle files.&quot;&quot;&quot;
    with Path(path).open(&quot;rb&quot;) as handle:
        return pickle.load(handle)


def save_torch_state(value: Any, path: Path | str) -&gt; Path:
    &quot;&quot;&quot;Atomically save torch state without importing torch until this function is used.&quot;&quot;&quot;
    return _write(value, path, &quot;torch&quot;)


@dataclass
class CompetitionConfig:
    &quot;&quot;&quot;Explicit notebook state; no installed project is required.&quot;&quot;&quot;

    data_dir: Path
    output_dir: Path
    seed: int = 42
    num_workers: int = 0
    pretrained_allowed: bool = False
    time_budget_minutes: float = 60
    device: str = &quot;auto&quot;

    def prepare(self) -&gt; None:
        self.device = (
            (&quot;cuda&quot; if torch.cuda.is_available() else &quot;cpu&quot;)
            if self.device == &quot;auto&quot;
            else self.device
        )
        self.output_dir.mkdir(parents=True, exist_ok=True)
        random.seed(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.seed)

    def to_dict(self) -&gt; dict:
        return {**asdict(self), &quot;data_dir&quot;: str(self.data_dir), &quot;output_dir&quot;: str(self.output_dir)}


class TimeBudget:
    &quot;&quot;&quot;Leave delivery time available; a running batch/fit cannot be interrupted safely.&quot;&quot;&quot;

    def __init__(self, minutes: float, reserve_minutes: float = 10) -&gt; None:
        if not 0 &lt;= reserve_minutes &lt; minutes:
            raise ValueError(&quot;Require 0 &lt;= reserve &lt; total minutes&quot;)
        self.deadline = time.monotonic() + 60 * (minutes - reserve_minutes)

    @property
    def should_stop(self) -&gt; bool:
        return time.monotonic() &gt;= self.deadline

    def checkpoint(self, stage: str) -&gt; None:
        if self.should_stop:
            raise TimeoutError(f&quot;Delivery reserve reached before {stage}&quot;)


def runtime_paths(
    *, runtime_dir: Path | None = None, persistent_dir: Path | None = None
) -&gt; SimpleNamespace:
    &quot;&quot;&quot;Media stays on local disk; optional durable storage is an explicit destination.&quot;&quot;&quot;
    root = (
        Path(runtime_dir)
        if runtime_dir is not None
        else (Path(&quot;/content/olp_runtime&quot;) if &quot;google.colab&quot; in sys.modules else Path.cwd())
    )
    data, output = root / &quot;data&quot;, root / &quot;outputs&quot;
    data.mkdir(parents=True, exist_ok=True)
    output.mkdir(parents=True, exist_ok=True)
    return SimpleNamespace(
        data_dir=data,
        output_dir=output,
        persistent_dir=Path(persistent_dir) if persistent_dir is not None else None,
    )


def mount_google_drive() -&gt; None:
    &quot;&quot;&quot;Call only when an explicitly configured Colab path uses Drive.&quot;&quot;&quot;
    from google.colab import drive

    drive.mount(&quot;/content/drive&quot;)


def stage_data(source: Path | str, destination: Path | str) -&gt; Path:
    &quot;&quot;&quot;Copy a folder or extract a ZIP once, then train from the local destination.&quot;&quot;&quot;
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if source == destination or destination.is_relative_to(source):
        raise ValueError(&quot;Staging destination must be separate from its source&quot;)
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError(&quot;Reuse staged DATA_DIR and set DATA_ARCHIVE=None&quot;)
    destination.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        with zipfile.ZipFile(source) as archive:
            for member in archive.infolist():
                if not (destination / member.filename).resolve().is_relative_to(destination):
                    raise ValueError(&quot;Archive contains a path outside the destination&quot;)
            archive.extractall(destination)
    return destination


def sync_artifacts(source: Path | str, destination: Path | str | None) -&gt; list[Path]:
    &quot;&quot;&quot;Copy the entire candidate, including encoders, preserving separate run folders.&quot;&quot;&quot;
    if destination is None:
        return []
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if (
        destination == source
        or destination.is_relative_to(source)
        or source.is_relative_to(destination)
    ):
        raise ValueError(&quot;Output and backup folders must be separate&quot;)
    copied = []
    for path in source.rglob(&quot;*&quot;):
        if path.is_file() and not path.name.endswith(&quot;.tmp&quot;):
            target = destination / path.relative_to(source)
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_name(f&quot;.{target.name}.{uuid4().hex}.tmp&quot;)
            try:
                shutil.copy2(path, temporary)
                temporary.replace(target)
            finally:
                temporary.unlink(missing_ok=True)
            copied.append(target)
    return copied


def gpu_report() -&gt; dict:
    &quot;&quot;&quot;Inspect the actual runtime before choosing image size, batch size or precision.&quot;&quot;&quot;
    if not torch.cuda.is_available():
        return {&quot;device&quot;: &quot;cpu&quot;, &quot;precision&quot;: &quot;float32&quot;}
    free, total = torch.cuda.mem_get_info()
    return {
        &quot;device&quot;: torch.cuda.get_device_name(),
        &quot;free_GiB&quot;: free / 2**30,
        &quot;total_GiB&quot;: total / 2**30,
        &quot;precision&quot;: &quot;bf16&quot; if torch.cuda.is_bf16_supported() else &quot;fp16&quot;,
    }


def evaluate_metric(name: str, truth: Any, prediction: Any, **kwargs: Any) -&gt; float:
    &quot;&quot;&quot;Only the metrics used in this lesson; task-specific graders appear beside the model.&quot;&quot;&quot;
    from sklearn.metrics import accuracy_score, roc_auc_score

    if name == &quot;roc_auc&quot;:
        return float(roc_auc_score(truth, prediction, **kwargs))
    if name == &quot;accuracy&quot;:
        return float(accuracy_score(truth, prediction))
    if name == &quot;dice_mean_image&quot;:
        y, p = np.asarray(truth, bool), np.asarray(prediction, bool)
        if y.shape != p.shape or not y.size:
            raise ValueError(&quot;Need aligned nonempty masks&quot;)
        axes = tuple(range(1, y.ndim))
        denominator = y.sum(axis=axes) + p.sum(axis=axes)
        return float(
            np.divide(
                2 * (y &amp; p).sum(axis=axes), denominator, out=np.ones(len(y)), where=denominator != 0
            ).mean()
        )
    raise ValueError(f&quot;Unsupported lesson metric: {name}&quot;)


FAMILY, VARIANT = &quot;07_matching&quot;, &quot;baseline&quot;
DATA_ARCHIVE, PERSISTENT_DIR = None, None
if &quot;google.colab&quot; in sys.modules and any(
    &quot;/content/drive/&quot; in str(value) for value in (DATA_ARCHIVE, PERSISTENT_DIR)
):
    mount_google_drive()
paths = runtime_paths(persistent_dir=PERSISTENT_DIR)
DATA_DIR = (
    paths.data_dir / &quot;learning&quot; / FAMILY
    if &quot;google.colab&quot; in sys.modules
    else Path.cwd() / &quot;data&quot; / &quot;learning&quot; / FAMILY
)
if DATA_ARCHIVE is not None:
    stage_data(DATA_ARCHIVE, DATA_DIR)
RUN_ID = f&quot;{FAMILY}-{VARIANT}-{uuid4().hex[:8]}&quot;
OUTPUT = paths.output_dir / &quot;learning&quot; / RUN_ID
DURABLE_OUTPUT = paths.persistent_dir / RUN_ID if paths.persistent_dir is not None else None
PRETRAINED_ALLOWED, WEIGHT_PATH = False, None
SEED, MAX_ROWS, IMAGE_SIZE, BATCH_SIZE, EPOCHS = 42, 6000, 128, 16, 3
TIME_BUDGET_MINUTES, DELIVERY_RESERVE_MINUTES = 60, 10
GROUP_COLUMN, TIME_COLUMN = &quot;label_group&quot;, None
USE_IMBALANCE_WEIGHTS = False
USE_HISTORY_FEATURES = False
MODEL_NAME = &quot;resnet18&quot;
IMAGE_SUFFIX = &quot;.png&quot;
TEXT_MODEL_DIR, IMAGE_MODEL_CHECKPOINT, EMBEDDING_ARCHIVE = None, None, None
ASSET_PROVENANCE = (
    &quot;&quot;  # exact asset/version/license; required when enabling local pretrained features
)
EMBEDDING_PATH, EMBEDDING_DIM, MAX_LENGTH = None, 64, 128
THRESHOLD_GRID = (0.3, 0.4, 0.5, 0.6, 0.7)
competition = CompetitionConfig(
    data_dir=DATA_DIR,
    output_dir=OUTPUT,
    seed=SEED,
    num_workers=0,
    pretrained_allowed=PRETRAINED_ALLOWED,
    time_budget_minutes=TIME_BUDGET_MINUTES,
)
competition.prepare()
budget = TimeBudget(TIME_BUDGET_MINUTES, reserve_minutes=DELIVERY_RESERVE_MINUTES)
print(gpu_report())
print(&quot;Local data:&quot;, DATA_DIR, &quot;New candidate:&quot;, OUTPUT)

</code></pre>
</details>


## 1a. Download the practice dataset (optional, before the timed exercise)

The cell is skipped by default. For online preparation, first accept this competition's
rules on its Kaggle page and configure credentials using the
[official authentication guide](https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication).
Set DOWNLOAD_DATA=True below. Local uv sync includes the official Kaggle client. In a copied
notebook/Colab runtime without it, install only `kaggle>=2.2,<3` before running this cell;
do not install this project or replace Colab's torch/NumPy/CUDA stack.

Use KAGGLE_API_TOKEN in your environment, the client's configured token/legacy credential
file, or enable PROMPT_FOR_KAGGLE_TOKEN for hidden input. Never paste a token into saved code.
No rules are accepted automatically, no submission is uploaded, and no weights are fetched.
Downloads can be several GB: allow space for the cached ZIP plus extracted data. DATA_DIR
must be a dedicated empty folder (or a completed download from this cell). Existing manual
data/fixtures are left untouched; keep DOWNLOAD_DATA=False to use them.

This cell reuses a completed dataset/cache, expands nested train/test/CSV archives, and
checks the expected layout. On Kaggle itself, attach competition data in the notebook UI,
point DATA_DIR at its mounted folder, and leave this switch off. For restricted-room drills,
prepare data beforehand and keep this cell disabled. The training timer restarts afterward.




In [ ]:
import os
import subprocess
from getpass import getpass
from importlib.util import find_spec

DOWNLOAD_DATA = False
KAGGLE_COMPETITION = "shopee-product-matching"
EXPECTED_DATA_PATHS = (
    "train.csv",
    "test.csv",
    "sample_submission.csv",
    "train_images",
    "test_images",
)
DOWNLOAD_CACHE = DATA_DIR.parent / "_downloads" / KAGGLE_COMPETITION
DOWNLOAD_TIMEOUT_SECONDS = 7200
PROMPT_FOR_KAGGLE_TOKEN = "google.colab" in sys.modules


def extract_dataset_zip(source: Path, destination: Path) -> None:
    """Extract data without allowing absolute paths, traversal or silent overwrites."""
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source) as archive:
        members = archive.infolist()
        if sum(member.file_size for member in members) > shutil.disk_usage(destination).free:
            raise OSError("Insufficient disk space for the next archive")
        for member in members:
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError("Archive contains a path outside the dataset folder")
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            # Duplicated paths indicate a wrong archive/layout; preserve the first file.
            with archive.open(member) as incoming, target.open("xb") as outgoing:
                shutil.copyfileobj(incoming, outgoing)


def download_practice_data(
    competition: str,
    destination: Path,
    cache: Path,
    expected_paths: Sequence[str],
    *,
    prompt_token: bool = False,
    timeout_seconds: int = 7200,
) -> Path:
    """Use the official CLI, then publish the complete extracted folder by renaming it."""
    if not competition or any(
        char not in "abcdefghijklmnopqrstuvwxyz0123456789-" for char in competition
    ):
        raise ValueError("Expected a Kaggle competition slug, not a URL or shell command")
    destination, cache = Path(destination).resolve(), Path(cache).resolve()
    if (
        destination == cache
        or cache.is_relative_to(destination)
        or destination.is_relative_to(cache)
    ):
        raise ValueError("Dataset and download cache must be separate folders")
    marker = destination / ".kaggle-source.json"
    if marker.is_file():
        source = json.loads(marker.read_text(encoding="utf-8"))
        if source.get("competition") == competition and all(
            (destination / path).exists() for path in expected_paths
        ):
            print("Reusing completed dataset:", destination)
            return destination
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError(
            "DATA_DIR is not empty/completed; choose a new folder or keep DOWNLOAD_DATA=False"
        )
    cache.mkdir(parents=True, exist_ok=True)
    bundle = cache / f"{competition}.zip"
    if not bundle.is_file() or not zipfile.is_zipfile(bundle):
        if find_spec("kaggle") is None:
            raise ModuleNotFoundError(
                "Install kaggle>=2.2,<3 for online preparation; "
                "local users can run uv sync --locked"
            )
        if prompt_token and not os.environ.get("KAGGLE_API_TOKEN"):
            token = getpass("Kaggle API token (blank uses configured credentials): ").strip()
            if token:
                os.environ["KAGGLE_API_TOKEN"] = token
        # No shell, embedded credentials, auto-acceptance, uploads or package installation.
        command = [
            sys.executable,
            "-m",
            "kaggle",
            "competitions",
            "download",
            competition,
            "--path",
            str(cache),
            "--quiet",
        ]
        if bundle.exists():
            command.append("--force")  # A partial/invalid archive must be fetched again.
        result = subprocess.run(
            command,
            stdin=subprocess.DEVNULL,
            capture_output=True,
            timeout=timeout_seconds,
            check=False,
        )
        if result.returncode:
            # Do not put client output, possible signed URLs or credentials in saved notebooks.
            raise RuntimeError(
                f"Kaggle download failed (exit {result.returncode}). "
                "Check credentials, competition access/rules, internet and disk space. "
                "No dataset was published."
            )
        if not bundle.is_file() or not zipfile.is_zipfile(bundle):
            raise ValueError("Kaggle did not produce a complete competition ZIP; check the cache")
    destination.parent.mkdir(parents=True, exist_ok=True)
    staging = destination.with_name(f".{destination.name}-incoming-{uuid4().hex[:8]}")
    staging.mkdir()
    try:
        extract_dataset_zip(bundle, staging)
        # TGS has train.zip with images/ and masks/; some tasks also wrap CSVs in ZIPs.
        for nested in sorted(staging.rglob("*.zip")):
            folder = nested.parent
            if nested.stem in {"train", "test", "train_images", "test_images"}:
                with zipfile.ZipFile(nested) as archive:
                    names = [item.filename for item in archive.infolist() if not item.is_dir()]
                if names and not all(name.startswith(nested.stem + "/") for name in names):
                    folder = nested.parent / nested.stem
            extract_dataset_zip(nested, folder)
            nested.unlink()  # The outer cached bundle remains available for recovery.
        missing = [path for path in expected_paths if not (staging / path).exists()]
        if missing:
            raise FileNotFoundError(f"Unexpected dataset layout; missing: {missing}")
        save_json(
            {
                "competition": competition,
                "source": f"https://www.kaggle.com/competitions/{competition}/data",
                "expected_paths": list(expected_paths),
            },
            staging / marker.name,
        )
        if destination.exists():
            destination.rmdir()  # Only the previously checked empty directory.
        staging.replace(destination)
    except Exception as error:
        raise RuntimeError(
            "Dataset preparation failed; DATA_DIR was not replaced. "
            f"Inspect partial extraction at {staging}"
        ) from error
    print("Prepared dataset:", destination)
    return destination


if DOWNLOAD_DATA:
    if DATA_ARCHIVE is not None:
        raise ValueError("Choose either DATA_ARCHIVE staging or DOWNLOAD_DATA, then rerun setup")
    DATA_DIR = download_practice_data(
        KAGGLE_COMPETITION,
        DATA_DIR,
        DOWNLOAD_CACHE,
        EXPECTED_DATA_PATHS,
        prompt_token=PROMPT_FOR_KAGGLE_TOKEN,
        timeout_seconds=DOWNLOAD_TIMEOUT_SECONDS,
    )
source_marker = DATA_DIR / ".kaggle-source.json"
if source_marker.is_file():
    save_json(json.loads(source_marker.read_text(encoding="utf-8")), OUTPUT / "data_source.json")
print("Download enabled:", DOWNLOAD_DATA, "Dataset folder:", DATA_DIR)
# Dataset acquisition is preparation time, not part of the measured training/delivery budget.
budget = TimeBudget(TIME_BUDGET_MINUTES, reserve_minutes=DELIVERY_RESERVE_MINUTES)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>import os
import subprocess
from getpass import getpass
from importlib.util import find_spec

DOWNLOAD_DATA = False
KAGGLE_COMPETITION = &quot;shopee-product-matching&quot;
EXPECTED_DATA_PATHS = (
    &quot;train.csv&quot;,
    &quot;test.csv&quot;,
    &quot;sample_submission.csv&quot;,
    &quot;train_images&quot;,
    &quot;test_images&quot;,
)
DOWNLOAD_CACHE = DATA_DIR.parent / &quot;_downloads&quot; / KAGGLE_COMPETITION
DOWNLOAD_TIMEOUT_SECONDS = 7200
PROMPT_FOR_KAGGLE_TOKEN = &quot;google.colab&quot; in sys.modules


def extract_dataset_zip(source: Path, destination: Path) -&gt; None:
    &quot;&quot;&quot;Extract data without allowing absolute paths, traversal or silent overwrites.&quot;&quot;&quot;
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source) as archive:
        members = archive.infolist()
        if sum(member.file_size for member in members) &gt; shutil.disk_usage(destination).free:
            raise OSError(&quot;Insufficient disk space for the next archive&quot;)
        for member in members:
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(&quot;Archive contains a path outside the dataset folder&quot;)
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            # Duplicated paths indicate a wrong archive/layout; preserve the first file.
            with archive.open(member) as incoming, target.open(&quot;xb&quot;) as outgoing:
                shutil.copyfileobj(incoming, outgoing)


def download_practice_data(
    competition: str,
    destination: Path,
    cache: Path,
    expected_paths: Sequence[str],
    *,
    prompt_token: bool = False,
    timeout_seconds: int = 7200,
) -&gt; Path:
    &quot;&quot;&quot;Use the official CLI, then publish the complete extracted folder by renaming it.&quot;&quot;&quot;
    if not competition or any(
        char not in &quot;abcdefghijklmnopqrstuvwxyz0123456789-&quot; for char in competition
    ):
        raise ValueError(&quot;Expected a Kaggle competition slug, not a URL or shell command&quot;)
    destination, cache = Path(destination).resolve(), Path(cache).resolve()
    if (
        destination == cache
        or cache.is_relative_to(destination)
        or destination.is_relative_to(cache)
    ):
        raise ValueError(&quot;Dataset and download cache must be separate folders&quot;)
    marker = destination / &quot;.kaggle-source.json&quot;
    if marker.is_file():
        source = json.loads(marker.read_text(encoding=&quot;utf-8&quot;))
        if source.get(&quot;competition&quot;) == competition and all(
            (destination / path).exists() for path in expected_paths
        ):
            print(&quot;Reusing completed dataset:&quot;, destination)
            return destination
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError(
            &quot;DATA_DIR is not empty/completed; choose a new folder or keep DOWNLOAD_DATA=False&quot;
        )
    cache.mkdir(parents=True, exist_ok=True)
    bundle = cache / f&quot;{competition}.zip&quot;
    if not bundle.is_file() or not zipfile.is_zipfile(bundle):
        if find_spec(&quot;kaggle&quot;) is None:
            raise ModuleNotFoundError(
                &quot;Install kaggle&gt;=2.2,&lt;3 for online preparation; &quot;
                &quot;local users can run uv sync --locked&quot;
            )
        if prompt_token and not os.environ.get(&quot;KAGGLE_API_TOKEN&quot;):
            token = getpass(&quot;Kaggle API token (blank uses configured credentials): &quot;).strip()
            if token:
                os.environ[&quot;KAGGLE_API_TOKEN&quot;] = token
        # No shell, embedded credentials, auto-acceptance, uploads or package installation.
        command = [
            sys.executable,
            &quot;-m&quot;,
            &quot;kaggle&quot;,
            &quot;competitions&quot;,
            &quot;download&quot;,
            competition,
            &quot;--path&quot;,
            str(cache),
            &quot;--quiet&quot;,
        ]
        if bundle.exists():
            command.append(&quot;--force&quot;)  # A partial/invalid archive must be fetched again.
        result = subprocess.run(
            command,
            stdin=subprocess.DEVNULL,
            capture_output=True,
            timeout=timeout_seconds,
            check=False,
        )
        if result.returncode:
            # Do not put client output, possible signed URLs or credentials in saved notebooks.
            raise RuntimeError(
                f&quot;Kaggle download failed (exit {result.returncode}). &quot;
                &quot;Check credentials, competition access/rules, internet and disk space. &quot;
                &quot;No dataset was published.&quot;
            )
        if not bundle.is_file() or not zipfile.is_zipfile(bundle):
            raise ValueError(&quot;Kaggle did not produce a complete competition ZIP; check the cache&quot;)
    destination.parent.mkdir(parents=True, exist_ok=True)
    staging = destination.with_name(f&quot;.{destination.name}-incoming-{uuid4().hex[:8]}&quot;)
    staging.mkdir()
    try:
        extract_dataset_zip(bundle, staging)
        # TGS has train.zip with images/ and masks/; some tasks also wrap CSVs in ZIPs.
        for nested in sorted(staging.rglob(&quot;*.zip&quot;)):
            folder = nested.parent
            if nested.stem in {&quot;train&quot;, &quot;test&quot;, &quot;train_images&quot;, &quot;test_images&quot;}:
                with zipfile.ZipFile(nested) as archive:
                    names = [item.filename for item in archive.infolist() if not item.is_dir()]
                if names and not all(name.startswith(nested.stem + &quot;/&quot;) for name in names):
                    folder = nested.parent / nested.stem
            extract_dataset_zip(nested, folder)
            nested.unlink()  # The outer cached bundle remains available for recovery.
        missing = [path for path in expected_paths if not (staging / path).exists()]
        if missing:
            raise FileNotFoundError(f&quot;Unexpected dataset layout; missing: {missing}&quot;)
        save_json(
            {
                &quot;competition&quot;: competition,
                &quot;source&quot;: f&quot;https://www.kaggle.com/competitions/{competition}/data&quot;,
                &quot;expected_paths&quot;: list(expected_paths),
            },
            staging / marker.name,
        )
        if destination.exists():
            destination.rmdir()  # Only the previously checked empty directory.
        staging.replace(destination)
    except Exception as error:
        raise RuntimeError(
            &quot;Dataset preparation failed; DATA_DIR was not replaced. &quot;
            f&quot;Inspect partial extraction at {staging}&quot;
        ) from error
    print(&quot;Prepared dataset:&quot;, destination)
    return destination


if DOWNLOAD_DATA:
    if DATA_ARCHIVE is not None:
        raise ValueError(&quot;Choose either DATA_ARCHIVE staging or DOWNLOAD_DATA, then rerun setup&quot;)
    DATA_DIR = download_practice_data(
        KAGGLE_COMPETITION,
        DATA_DIR,
        DOWNLOAD_CACHE,
        EXPECTED_DATA_PATHS,
        prompt_token=PROMPT_FOR_KAGGLE_TOKEN,
        timeout_seconds=DOWNLOAD_TIMEOUT_SECONDS,
    )
source_marker = DATA_DIR / &quot;.kaggle-source.json&quot;
if source_marker.is_file():
    save_json(json.loads(source_marker.read_text(encoding=&quot;utf-8&quot;)), OUTPUT / &quot;data_source.json&quot;)
print(&quot;Download enabled:&quot;, DOWNLOAD_DATA, &quot;Dataset folder:&quot;, DATA_DIR)
# Dataset acquisition is preparation time, not part of the measured training/delivery budget.
budget = TimeBudget(TIME_BUDGET_MINUTES, reserve_minutes=DELIVERY_RESERVE_MINUTES)

</code></pre>
</details>


## 2. Load data and enforce the contract

Check schema and identity before fitting. A row is not always a prediction unit: detection has
several annotation rows per image, and matching has several records per product.

Stage the original train/test CSVs and train_images/test_images directories. This matching
competition asks for a set of gallery posting IDs per query. The test gallery is used for
pairwise inference; no test labels or pseudo-label training are used.




In [ ]:
ID_COLUMN, TARGET_COLUMNS = "posting_id", ["matches"]
train = pd.read_csv(DATA_DIR / "train.csv", converters={ID_COLUMN: str, "label_group": str})
test = pd.read_csv(DATA_DIR / "test.csv", converters={ID_COLUMN: str})
sample = pd.read_csv(DATA_DIR / "sample_submission.csv", converters={ID_COLUMN: str})
train["title"] = train["title"].fillna("")
test["title"] = test["title"].fillna("")
train["_image"] = "train_images/" + train["image"]
test["_image"] = "test_images/" + test["image"]

if MAX_ROWS is not None and len(train) > MAX_ROWS:
    group_sizes = train.groupby("label_group").size().sample(frac=1, random_state=SEED)
    # Keep whole groups, including the group that crosses the approximate row budget.
    keep_groups = group_sizes.index[group_sizes.cumsum().shift(fill_value=0) < MAX_ROWS]
    train = train.loc[train["label_group"].isin(keep_groups)].reset_index(drop=True)
for name, frame in (("train", train), ("test", test), ("sample", sample)):
    if (
        frame[ID_COLUMN].isna().any()
        or frame[ID_COLUMN].eq("").any()
        or frame[ID_COLUMN].duplicated().any()
    ):
        raise ValueError(f"{name} must have unique, nonempty prediction-unit IDs")
assert not set(train[ID_COLUMN]) & set(test[ID_COLUMN])
assert set(test[ID_COLUMN]) == set(sample[ID_COLUMN])
assert sample.columns.tolist() == [ID_COLUMN, *TARGET_COLUMNS]
print(train.head(3).to_string(index=False))
print("Shapes:", train.shape, test.shape, sample.shape)

if train["label_group"].eq("").any() or train["label_group"].isna().any():
    raise ValueError("Every training item needs a product-group identity")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>ID_COLUMN, TARGET_COLUMNS = &quot;posting_id&quot;, [&quot;matches&quot;]
train = pd.read_csv(DATA_DIR / &quot;train.csv&quot;, converters={ID_COLUMN: str, &quot;label_group&quot;: str})
test = pd.read_csv(DATA_DIR / &quot;test.csv&quot;, converters={ID_COLUMN: str})
sample = pd.read_csv(DATA_DIR / &quot;sample_submission.csv&quot;, converters={ID_COLUMN: str})
train[&quot;title&quot;] = train[&quot;title&quot;].fillna(&quot;&quot;)
test[&quot;title&quot;] = test[&quot;title&quot;].fillna(&quot;&quot;)
train[&quot;_image&quot;] = &quot;train_images/&quot; + train[&quot;image&quot;]
test[&quot;_image&quot;] = &quot;test_images/&quot; + test[&quot;image&quot;]

if MAX_ROWS is not None and len(train) &gt; MAX_ROWS:
    group_sizes = train.groupby(&quot;label_group&quot;).size().sample(frac=1, random_state=SEED)
    # Keep whole groups, including the group that crosses the approximate row budget.
    keep_groups = group_sizes.index[group_sizes.cumsum().shift(fill_value=0) &lt; MAX_ROWS]
    train = train.loc[train[&quot;label_group&quot;].isin(keep_groups)].reset_index(drop=True)
for name, frame in ((&quot;train&quot;, train), (&quot;test&quot;, test), (&quot;sample&quot;, sample)):
    if (
        frame[ID_COLUMN].isna().any()
        or frame[ID_COLUMN].eq(&quot;&quot;).any()
        or frame[ID_COLUMN].duplicated().any()
    ):
        raise ValueError(f&quot;{name} must have unique, nonempty prediction-unit IDs&quot;)
assert not set(train[ID_COLUMN]) &amp; set(test[ID_COLUMN])
assert set(test[ID_COLUMN]) == set(sample[ID_COLUMN])
assert sample.columns.tolist() == [ID_COLUMN, *TARGET_COLUMNS]
print(train.head(3).to_string(index=False))
print(&quot;Shapes:&quot;, train.shape, test.shape, sample.shape)

if train[&quot;label_group&quot;].eq(&quot;&quot;).any() or train[&quot;label_group&quot;].isna().any():
    raise ValueError(&quot;Every training item needs a product-group identity&quot;)

</code></pre>
</details>


## 3. Inspect and visualize the data

Write down three observations from these plots before revealing any solution: missingness,
examples that look wrong, and a possible leakage unit. Never select experiments using hidden
test labels.






In [ ]:
group_sizes = train.groupby("label_group").size()
print("Product group-size quantiles:", group_sizes.quantile([0, 0.5, 0.9, 1]).to_dict())
print("Singleton product fraction:", (group_sizes == 1).mean())
print("Repeated image hashes:", train["image_phash"].duplicated().sum())
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (_, row) in zip(axes, train.head(3).iterrows(), strict=False):
    with Image.open(DATA_DIR / row["_image"]) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(row["title"][:55])
    ax.axis("off")
fig.savefig(OUTPUT / "inspection.png")
plt.show()
plt.close(fig)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>group_sizes = train.groupby(&quot;label_group&quot;).size()
print(&quot;Product group-size quantiles:&quot;, group_sizes.quantile([0, 0.5, 0.9, 1]).to_dict())
print(&quot;Singleton product fraction:&quot;, (group_sizes == 1).mean())
print(&quot;Repeated image hashes:&quot;, train[&quot;image_phash&quot;].duplicated().sum())
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (_, row) in zip(axes, train.head(3).iterrows(), strict=False):
    with Image.open(DATA_DIR / row[&quot;_image&quot;]) as image:
        ax.imshow(image.convert(&quot;RGB&quot;))
    ax.set_title(row[&quot;title&quot;][:55])
    ax.axis(&quot;off&quot;)
fig.savefig(OUTPUT / &quot;inspection.png&quot;)
plt.show()
plt.close(fig)

</code></pre>
</details>


## 4. Freeze a leakage-aware split

Keep these exact folds for baseline/advanced comparisons. Random stratification is only
appropriate for independent samples. Set a group or time column when the acquisition process
demands it; assert the chosen boundary.

A product-group holdout estimates generalization to unseen products. A random item split can
expose the same product on both sides. This exercise always holds out label_group,
independently of the generic GROUP_COLUMN field.




In [ ]:
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, train_test_split


@dataclass(slots=True)
class SplitResult:
    """Training/validation tables plus their original positional indexes."""

    train: pd.DataFrame
    valid: pd.DataFrame
    train_indices: np.ndarray
    valid_indices: np.ndarray


def _stratification_key(frame: pd.DataFrame, targets: Sequence[str]) -> pd.Series:
    if not targets:
        raise ValueError("At least one target is required for stratification")
    if len(targets) == 1:
        return frame[targets[0]].astype(str)
    return frame[list(targets)].astype(str).agg("|".join, axis=1)


def make_split(
    frame: pd.DataFrame,
    *,
    target_columns: str | Sequence[str] | None = None,
    group_column: str | None = None,
    time_column: str | None = None,
    valid_size: float = 0.2,
    seed: int = 42,
) -> SplitResult:
    """Create a deterministic holdout with leakage-aware group and time options."""
    if not 0 < valid_size < 1:
        raise ValueError("valid_size must be between 0 and 1")
    if len(frame) < 2:
        raise ValueError("At least two rows are required")
    targets = [target_columns] if isinstance(target_columns, str) else list(target_columns or [])
    for column in [*targets, group_column, time_column]:
        if column is not None and column not in frame.columns:
            raise KeyError(f"Missing split column: {column}")

    indices = np.arange(len(frame))
    if time_column:
        timestamps = pd.to_datetime(frame[time_column], errors="raise")
        if timestamps.isna().any():
            raise ValueError("Time split requires complete timestamps")
        order = np.argsort(timestamps.to_numpy(), kind="stable")
        ordered_times = timestamps.to_numpy()[order]
        boundaries = np.flatnonzero(ordered_times[1:] != ordered_times[:-1]) + 1
        if not len(boundaries):
            raise ValueError("Need at least two distinct times for a strict chronological split")
        desired = len(frame) * (1 - valid_size)
        cut = int(boundaries[np.argmin(abs(boundaries - desired))])
        train_idx, valid_idx = order[:cut], order[cut:]
    elif group_column and targets:
        folds = max(2, round(1 / valid_size))
        splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
        try:
            train_idx, valid_idx = next(
                splitter.split(indices, _stratification_key(frame, targets), frame[group_column])
            )
        except ValueError:
            fallback = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=seed)
            train_idx, valid_idx = next(fallback.split(indices, groups=frame[group_column]))
    elif group_column:
        splitter = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=seed)
        train_idx, valid_idx = next(splitter.split(indices, groups=frame[group_column]))
    else:
        stratify = _stratification_key(frame, targets) if targets else None
        try:
            train_idx, valid_idx = train_test_split(
                indices, test_size=valid_size, random_state=seed, stratify=stratify
            )
        except ValueError:
            train_idx, valid_idx = train_test_split(
                indices, test_size=valid_size, random_state=seed, stratify=None
            )

    if group_column:
        overlap = set(frame.iloc[train_idx][group_column]) & set(
            frame.iloc[valid_idx][group_column]
        )
        if overlap:
            raise RuntimeError(f"Group leakage detected for {len(overlap)} groups")
    return SplitResult(
        train=frame.iloc[train_idx].reset_index(drop=True),
        valid=frame.iloc[valid_idx].reset_index(drop=True),
        train_indices=np.asarray(train_idx),
        valid_indices=np.asarray(valid_idx),
    )


if TIME_COLUMN:
    raise ValueError("Use group holdout here; design temporal gallery evaluation separately")
raise NotImplementedError('TODO: hold out complete product groups, not random pairs or random items')
assert not set(tr["label_group"]) & set(va["label_group"])
assert not set(tr[ID_COLUMN]) & set(va[ID_COLUMN])
print("Held-out product groups:", va["label_group"].nunique(), "Gallery items:", len(va))



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, train_test_split


@dataclass(slots=True)
class SplitResult:
    &quot;&quot;&quot;Training/validation tables plus their original positional indexes.&quot;&quot;&quot;

    train: pd.DataFrame
    valid: pd.DataFrame
    train_indices: np.ndarray
    valid_indices: np.ndarray


def _stratification_key(frame: pd.DataFrame, targets: Sequence[str]) -&gt; pd.Series:
    if not targets:
        raise ValueError(&quot;At least one target is required for stratification&quot;)
    if len(targets) == 1:
        return frame[targets[0]].astype(str)
    return frame[list(targets)].astype(str).agg(&quot;|&quot;.join, axis=1)


def make_split(
    frame: pd.DataFrame,
    *,
    target_columns: str | Sequence[str] | None = None,
    group_column: str | None = None,
    time_column: str | None = None,
    valid_size: float = 0.2,
    seed: int = 42,
) -&gt; SplitResult:
    &quot;&quot;&quot;Create a deterministic holdout with leakage-aware group and time options.&quot;&quot;&quot;
    if not 0 &lt; valid_size &lt; 1:
        raise ValueError(&quot;valid_size must be between 0 and 1&quot;)
    if len(frame) &lt; 2:
        raise ValueError(&quot;At least two rows are required&quot;)
    targets = [target_columns] if isinstance(target_columns, str) else list(target_columns or [])
    for column in [*targets, group_column, time_column]:
        if column is not None and column not in frame.columns:
            raise KeyError(f&quot;Missing split column: {column}&quot;)

    indices = np.arange(len(frame))
    if time_column:
        timestamps = pd.to_datetime(frame[time_column], errors=&quot;raise&quot;)
        if timestamps.isna().any():
            raise ValueError(&quot;Time split requires complete timestamps&quot;)
        order = np.argsort(timestamps.to_numpy(), kind=&quot;stable&quot;)
        ordered_times = timestamps.to_numpy()[order]
        boundaries = np.flatnonzero(ordered_times[1:] != ordered_times[:-1]) + 1
        if not len(boundaries):
            raise ValueError(&quot;Need at least two distinct times for a strict chronological split&quot;)
        desired = len(frame) * (1 - valid_size)
        cut = int(boundaries[np.argmin(abs(boundaries - desired))])
        train_idx, valid_idx = order[:cut], order[cut:]
    elif group_column and targets:
        folds = max(2, round(1 / valid_size))
        splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
        try:
            train_idx, valid_idx = next(
                splitter.split(indices, _stratification_key(frame, targets), frame[group_column])
            )
        except ValueError:
            fallback = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=seed)
            train_idx, valid_idx = next(fallback.split(indices, groups=frame[group_column]))
    elif group_column:
        splitter = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=seed)
        train_idx, valid_idx = next(splitter.split(indices, groups=frame[group_column]))
    else:
        stratify = _stratification_key(frame, targets) if targets else None
        try:
            train_idx, valid_idx = train_test_split(
                indices, test_size=valid_size, random_state=seed, stratify=stratify
            )
        except ValueError:
            train_idx, valid_idx = train_test_split(
                indices, test_size=valid_size, random_state=seed, stratify=None
            )

    if group_column:
        overlap = set(frame.iloc[train_idx][group_column]) &amp; set(
            frame.iloc[valid_idx][group_column]
        )
        if overlap:
            raise RuntimeError(f&quot;Group leakage detected for {len(overlap)} groups&quot;)
    return SplitResult(
        train=frame.iloc[train_idx].reset_index(drop=True),
        valid=frame.iloc[valid_idx].reset_index(drop=True),
        train_indices=np.asarray(train_idx),
        valid_indices=np.asarray(valid_idx),
    )


if TIME_COLUMN:
    raise ValueError(&quot;Use group holdout here; design temporal gallery evaluation separately&quot;)
split = make_split(train, group_column=&quot;label_group&quot;, valid_size=0.2, seed=SEED)
tr, va = split.train.copy(), split.valid.copy()
assert not set(tr[&quot;label_group&quot;]) &amp; set(va[&quot;label_group&quot;])
assert not set(tr[ID_COLUMN]) &amp; set(va[ID_COLUMN])
print(&quot;Held-out product groups:&quot;, va[&quot;label_group&quot;].nunique(), &quot;Gallery items:&quot;, len(va))

</code></pre>
</details>


## 5. Audit imbalance and choose a response

Measure prevalence on the training fold. Choose the response according to the official score:
class weights, positive weights, coverage sampling or a log target are hypotheses. Compare
with the unweighted candidate on the same split. Never rebalance validation.

Most arbitrary item pairs are negatives. Pairwise accuracy would reward predicting no matches.
Mean per-query set F1 and group-aware validation match the actual decision problem; a later
supervised metric-learning sampler should reflect product identities.




In [ ]:
raise NotImplementedError('TODO: measure group-size imbalance, which determines positive/negative pair imbalance')
print("Positive pairs:", positive_pairs, "All pairs:", all_pairs)
print("Response: group holdout and per-query F1; keep validation unchanged")
print("For metric learning later: sample product groups, not all pairs equally.")
sizes.hist(bins=20)
plt.title("Training product-group sizes")
plt.savefig(OUTPUT / "imbalance.png")
plt.show()
plt.close()



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>sizes = tr.groupby(&quot;label_group&quot;).size()
positive_pairs = int((sizes * (sizes - 1) / 2).sum())
all_pairs = len(tr) * (len(tr) - 1) // 2
print(&quot;Positive pairs:&quot;, positive_pairs, &quot;All pairs:&quot;, all_pairs)
print(&quot;Response: group holdout and per-query F1; keep validation unchanged&quot;)
print(&quot;For metric learning later: sample product groups, not all pairs equally.&quot;)
sizes.hist(bins=20)
plt.title(&quot;Training product-group sizes&quot;)
plt.savefig(OUTPUT / &quot;imbalance.png&quot;)
plt.show()
plt.close()

</code></pre>
</details>


## 6. Build features or datasets

Inspect one transformed example or batch. Explain its shape, range, label encoding and
train/validation/test consistency. Fitted preprocessing sees the training fold only.

Titles are transformed using a vocabulary fitted only on training products. Unknown validation
terms are an expected limitation, not a reason to fit on validation/test.




In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

raise NotImplementedError('TODO: use a training-title vocabulary; test is transformed, never fitted')
print("Training title preview:", tr["title"].head(3).tolist())



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_features=60000, sublinear_tf=True)
print(&quot;Training title preview:&quot;, tr[&quot;title&quot;].head(3).tolist())

</code></pre>
</details>


## 7. Define the model and loss

State why the output dimension and loss fit this prediction contract. First verify the
simplest model. Pretrained initialization is disabled unless the exact local asset is
permitted.

The similarity calculation uses query batches rather than an N-by-N dense allocation. Its
output can still be large if the threshold returns nearly every item; inspect match-count
distributions.




In [ ]:
from numpy.typing import ArrayLike
from sklearn.metrics.pairwise import cosine_similarity


def matching_f1(groups: Sequence[object], matches: Sequence[Sequence[int]]) -> float:
    """Shopee mean per-query set F1, including the query itself in its group and prediction."""
    labels = np.asarray(groups)
    if not len(labels) or len(labels) != len(matches):
        raise ValueError("Provide one match set per gallery item")
    values = []
    for index, predictions in enumerate(matches):
        candidate = set(predictions)
        if not candidate or any(not 0 <= item < len(labels) for item in candidate):
            raise ValueError("Match sets must contain valid gallery positions")
        relevant = set(np.flatnonzero(labels == labels[index]).tolist())
        values.append(2 * len(candidate & relevant) / (len(candidate) + len(relevant)))
    return float(np.mean(values))


def match_positions(features, threshold, batch_size=32):
    if not 0 < threshold <= 1:
        raise ValueError("Choose a cosine threshold in (0, 1]")
    matches = []
    raise NotImplementedError('TODO: bound the dense query-by-gallery matrix and always include self')
    return matches


def features_for(frame, weight=0.0):
    return vectorizer.transform(frame["title"])


MODEL_DESCRIPTION = "training-title TF-IDF cosine matching"



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>from numpy.typing import ArrayLike
from sklearn.metrics.pairwise import cosine_similarity


def matching_f1(groups: Sequence[object], matches: Sequence[Sequence[int]]) -&gt; float:
    &quot;&quot;&quot;Shopee mean per-query set F1, including the query itself in its group and prediction.&quot;&quot;&quot;
    labels = np.asarray(groups)
    if not len(labels) or len(labels) != len(matches):
        raise ValueError(&quot;Provide one match set per gallery item&quot;)
    values = []
    for index, predictions in enumerate(matches):
        candidate = set(predictions)
        if not candidate or any(not 0 &lt;= item &lt; len(labels) for item in candidate):
            raise ValueError(&quot;Match sets must contain valid gallery positions&quot;)
        relevant = set(np.flatnonzero(labels == labels[index]).tolist())
        values.append(2 * len(candidate &amp; relevant) / (len(candidate) + len(relevant)))
    return float(np.mean(values))


def match_positions(features, threshold, batch_size=32):
    if not 0 &lt; threshold &lt;= 1:
        raise ValueError(&quot;Choose a cosine threshold in (0, 1]&quot;)
    matches = []
    for start in range(0, features.shape[0], batch_size):
        similarities = cosine_similarity(features[start : start + batch_size], features)
        for local, row in enumerate(similarities):
            row[start + local] = 1.0
            matches.append(np.flatnonzero(row &gt;= threshold).tolist())
    return matches


def features_for(frame, weight=0.0):
    return vectorizer.transform(frame[&quot;title&quot;])


MODEL_DESCRIPTION = &quot;training-title TF-IDF cosine matching&quot;

</code></pre>
</details>


## 8. Train and preserve the candidate

First fit a tiny subset as a debugging drill; rebuild before the real-fold experiment. Main
torch lessons define their small training loop directly below, including precision, time
reserves and atomic best checkpoints. For sklearn fit the full preprocessing pipeline together.

The fit operation learns vocabulary and IDF on training products. Cosine comparisons within
the validation/test gallery are inference under this original task's contract.




In [ ]:
started = perf_counter()
budget.checkpoint("fit training-title vocabulary")
raise NotImplementedError('TODO: fit on training products only and preserve that vocabulary')
fit_seconds = perf_counter() - started



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>started = perf_counter()
budget.checkpoint(&quot;fit training-title vocabulary&quot;)
vectorizer.fit(tr[&quot;title&quot;])
checkpoint_path = save_pickle(vectorizer, OUTPUT / &quot;vectorizer.pkl&quot;)
fit_seconds = perf_counter() - started

</code></pre>
</details>


## 9. Validate, inspect errors and compare cost

Use all validation predictions for the exact score. Inspect individual mistakes and
minority/group performance; the diagnostic metric does not replace the official metric. Keep a
score/runtime ledger before choosing more complexity.

The bounded grid chooses thresholds/mixture weights on validation only. This reuse makes tiny
gains optimistic; check a second product-group split when worthwhile. Preserve the baseline
comparison and include full embedding extraction/inference cost.




In [ ]:
selection_started = perf_counter()
weights_to_compare = (0.0,)
thresholds = (0.5,)
comparisons = []
for weight in weights_to_compare:
    for threshold in thresholds:
        budget.checkpoint("validation matching comparison")
        features = features_for(va, weight)
        positions = match_positions(features, threshold)
        raise NotImplementedError('TODO: score complete match sets including self, not pairwise accuracy')
        comparisons.append({"weight": weight, "threshold": threshold, "score": value})
print(
    "Self-only reference F1:",
    matching_f1(va["label_group"].tolist(), [[index] for index in range(len(va))]),
)
comparison = pd.DataFrame(comparisons)
chosen = comparison.sort_values("score", ascending=False, kind="stable").iloc[0]
selected_weight, selected_threshold = float(chosen["weight"]), float(chosen["threshold"])
validation_score = float(chosen["score"])
validation_positions = match_positions(features_for(va, selected_weight), selected_threshold)
fit_seconds += perf_counter() - selection_started
print("Validation-only comparison:", comparison.to_string(index=False))
comparison.to_csv(OUTPUT / "threshold_comparison.csv", index=False)
errors = pd.DataFrame(
    {
        ID_COLUMN: va[ID_COLUMN],
        "true_group": va["label_group"],
        "predicted_match_count": [len(value) for value in validation_positions],
    }
)
print(errors.head(20).to_string(index=False))
errors.to_csv(OUTPUT / "validation_errors.csv", index=False)
comparison.plot.scatter(
    x="threshold",
    y="score",
    c="weight",
    cmap="viridis",
    title="Validation F1 versus threshold; color = image weight",
)
plt.savefig(OUTPUT / "validation.png")
plt.show()
plt.close()



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>selection_started = perf_counter()
weights_to_compare = (0.0,)
thresholds = (0.5,)
comparisons = []
for weight in weights_to_compare:
    for threshold in thresholds:
        budget.checkpoint(&quot;validation matching comparison&quot;)
        features = features_for(va, weight)
        positions = match_positions(features, threshold)
        value = matching_f1(va[&quot;label_group&quot;].tolist(), positions)
        comparisons.append({&quot;weight&quot;: weight, &quot;threshold&quot;: threshold, &quot;score&quot;: value})
print(
    &quot;Self-only reference F1:&quot;,
    matching_f1(va[&quot;label_group&quot;].tolist(), [[index] for index in range(len(va))]),
)
comparison = pd.DataFrame(comparisons)
chosen = comparison.sort_values(&quot;score&quot;, ascending=False, kind=&quot;stable&quot;).iloc[0]
selected_weight, selected_threshold = float(chosen[&quot;weight&quot;]), float(chosen[&quot;threshold&quot;])
validation_score = float(chosen[&quot;score&quot;])
validation_positions = match_positions(features_for(va, selected_weight), selected_threshold)
fit_seconds += perf_counter() - selection_started
print(&quot;Validation-only comparison:&quot;, comparison.to_string(index=False))
comparison.to_csv(OUTPUT / &quot;threshold_comparison.csv&quot;, index=False)
errors = pd.DataFrame(
    {
        ID_COLUMN: va[ID_COLUMN],
        &quot;true_group&quot;: va[&quot;label_group&quot;],
        &quot;predicted_match_count&quot;: [len(value) for value in validation_positions],
    }
)
print(errors.head(20).to_string(index=False))
errors.to_csv(OUTPUT / &quot;validation_errors.csv&quot;, index=False)
comparison.plot.scatter(
    x=&quot;threshold&quot;,
    y=&quot;score&quot;,
    c=&quot;weight&quot;,
    cmap=&quot;viridis&quot;,
    title=&quot;Validation F1 versus threshold; color = image weight&quot;,
)
plt.savefig(OUTPUT / &quot;validation.png&quot;)
plt.show()
plt.close()

</code></pre>
</details>


## 10. Reload and run complete inference

Rebuild or reload the saved candidate before test inference. Preserve ID order through the
loader, then align to the sample by ID. Check that reloaded validation predictions agree. This
is inference recovery, not exact training continuation.






In [ ]:
started = perf_counter()
raise NotImplementedError('TODO: reload the training vocabulary; never refit it on the test gallery')
output_values = {
    "matches": [" ".join(test.iloc[positions][ID_COLUMN].tolist()) for positions in test_positions]
}
for index, positions in enumerate(test_positions):
    assert index in positions and len(positions) == len(set(positions))
save_json(
    {
        "threshold": selected_threshold,
        "vocabulary_fit": "training products only",
        "matching": "test-to-test gallery inference; no training on test",
    },
    OUTPUT / "recipe.json",
)
inference_seconds = perf_counter() - started



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>started = perf_counter()
vectorizer = load_pickle(checkpoint_path)
assert match_positions(features_for(va), selected_threshold) == validation_positions
test_positions = match_positions(features_for(test), selected_threshold)
output_values = {
    &quot;matches&quot;: [&quot; &quot;.join(test.iloc[positions][ID_COLUMN].tolist()) for positions in test_positions]
}
for index, positions in enumerate(test_positions):
    assert index in positions and len(positions) == len(set(positions))
save_json(
    {
        &quot;threshold&quot;: selected_threshold,
        &quot;vocabulary_fit&quot;: &quot;training products only&quot;,
        &quot;matching&quot;: &quot;test-to-test gallery inference; no training on test&quot;,
    },
    OUTPUT / &quot;recipe.json&quot;,
)
inference_seconds = perf_counter() - started

</code></pre>
</details>


## 11. Validate the submission and persist the recipe

Validate the on-disk UTF-8 file, expected columns, ID order and task-specific value
constraints. Keep source, preprocessing, fold IDs, labels, thresholds, score and runtime with
the selected artifact. A checked CSV is not proof of upload acceptance.







In [ ]:
@dataclass(slots=True)
class SubmissionValidation:
    """Validation errors and non-fatal warnings for a candidate submission."""

    valid: bool
    errors: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)

    def raise_for_errors(self) -> None:
        """Raise one ``ValueError`` containing every validation error."""
        if self.errors:
            raise ValueError("Invalid submission: " + "; ".join(self.errors))


def build_submission(
    sample: pd.DataFrame,
    predictions: Mapping[str, Sequence[object]],
    *,
    prediction_ids: pd.DataFrame | Sequence[object] | None = None,
    id_columns: str | Iterable[str] | None = None,
) -> pd.DataFrame:
    """Copy the sample and optionally align predictions using unique (possibly composite) IDs.

    Without IDs, predictions must already be in sample order. Prefer explicit IDs for inference.
    Alignment rejects missing, duplicate, extra, or absent keys rather than guessing row order.
    """
    submission = sample.copy()
    ids = [id_columns] if isinstance(id_columns, str) else list(id_columns or [])
    order = None
    if prediction_ids is not None:
        if not ids:
            raise ValueError("id_columns is required with prediction_ids")
        if isinstance(prediction_ids, pd.DataFrame):
            keys = prediction_ids[ids].reset_index(drop=True)
        elif len(ids) == 1:
            keys = pd.DataFrame({ids[0]: list(prediction_ids)})
        else:
            raise ValueError("Composite prediction IDs must be a DataFrame")
        for name, frame in (("sample", sample), ("predictions", keys)):
            if (
                frame[ids].isna().any().any()
                or frame[ids].eq("").any().any()
                or frame.duplicated(ids).any()
            ):
                raise ValueError(f"{name} IDs must be nonmissing and unique across {ids}")
        expected = pd.MultiIndex.from_frame(sample[ids])
        actual = pd.MultiIndex.from_frame(keys)
        order = actual.get_indexer(expected)
        if len(actual) != len(expected) or (order < 0).any():
            raise ValueError("Prediction IDs must match the sample IDs exactly")
    for column, values in predictions.items():
        if column not in submission.columns:
            raise KeyError(f"Prediction column not present in sample submission: {column}")
        if len(values) != len(submission):
            raise ValueError(
                f"Column {column} has {len(values)} values; expected {len(submission)}"
            )
        if column in ids:
            raise ValueError(f"Cannot overwrite ID column: {column}")
        values = list(values)
        submission[column] = values if order is None else [values[index] for index in order]
    return submission


def validate_submission(
    submission: pd.DataFrame,
    sample: pd.DataFrame,
    *,
    id_columns: str | Iterable[str] | None = None,
    allowed_labels: Mapping[str, Iterable[object]] | None = None,
) -> SubmissionValidation:
    """Check schema, row count, ID order, missing values, and optional allowed labels."""
    errors: list[str] = []
    warnings: list[str] = []
    if submission.columns.duplicated().any() or sample.columns.duplicated().any():
        errors.append("Submission and sample column names must be unique")
        return SubmissionValidation(valid=False, errors=errors)
    if list(submission.columns) != list(sample.columns):
        errors.append(
            f"Columns/order differ: got {list(submission.columns)}, expected {list(sample.columns)}"
        )
    if len(submission) != len(sample):
        errors.append(f"Row count differs: got {len(submission)}, expected {len(sample)}")
    if submission.isna().any().any():
        columns = submission.columns[submission.isna().any()].tolist()
        errors.append(f"Missing values in columns: {columns}")
    numeric = submission.select_dtypes(include="number")
    if not np.isfinite(numeric.to_numpy(dtype=float, na_value=np.nan)).all():
        errors.append("Non-finite numeric values are not allowed")
    ids = [id_columns] if isinstance(id_columns, str) else list(id_columns or [])
    for column in ids:
        if column not in submission or column not in sample:
            errors.append(f"Missing ID column: {column}")
        elif submission[column].isna().any() or submission[column].eq("").any():
            errors.append(f"Missing ID values in column: {column}")
        elif len(submission) == len(sample) and not submission[column].equals(sample[column]):
            errors.append(f"ID values/order changed in column: {column}")
    if ids and all(column in submission for column in ids) and submission.duplicated(ids).any():
        warnings.append(f"Duplicate ID keys across columns: {ids}")
    for column, labels in (allowed_labels or {}).items():
        if column not in submission:
            errors.append(f"Missing allowed-label column: {column}")
        else:
            invalid = set(submission[column].dropna().unique()) - set(labels)
            if invalid:
                errors.append(f"Unexpected labels in {column}: {sorted(map(str, invalid))[:10]}")
    return SubmissionValidation(valid=not errors, errors=errors, warnings=warnings)


def write_submission(
    submission: pd.DataFrame,
    path: Path | str,
    *,
    sample: pd.DataFrame | None = None,
    id_columns: str | Iterable[str] | None = None,
    allowed_labels: Mapping[str, Iterable[object]] | None = None,
) -> Path:
    """Validate then atomically write a UTF-8 CSV submission."""
    if sample is not None:
        validation = validate_submission(
            submission, sample, id_columns=id_columns, allowed_labels=allowed_labels
        )
        validation.raise_for_errors()
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f".{destination.name}.{uuid4().hex}.tmp")
    try:
        submission.to_csv(temporary, index=False, encoding="utf-8")
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination


def _mapping(value: Any) -> Mapping[str, Any]:
    if is_dataclass(value):
        return asdict(value)
    if isinstance(value, Mapping):
        return value
    raise TypeError("Expected a dataclass or mapping")


def technical_report(
    *,
    task: str,
    model: str,
    data_summary: Mapping[str, Any],
    split_summary: Mapping[str, Any],
    training_config: Any,
    validation_metrics: Mapping[str, float],
    augmentations: list[str] | None = None,
    checkpoint: str | None = None,
    notes: list[str] | None = None,
) -> str:
    """Render the standard competition handoff report as Markdown text."""
    config = _mapping(training_config)
    lines = [
        "# Technical Report",
        "",
        f"- Task: {task}",
        f"- Model: {model}",
        f"- Checkpoint: {checkpoint or 'not recorded'}",
        "",
        "## Data and validation",
        "",
    ]
    lines.extend(f"- {key}: {value}" for key, value in data_summary.items())
    lines.extend(f"- Split {key}: {value}" for key, value in split_summary.items())
    lines.extend(["", "## Training", ""])
    lines.extend(f"- {key}: {value}" for key, value in config.items())
    lines.extend(["", "## Validation metrics", ""])
    lines.extend(f"- {key}: {value:.6f}" for key, value in validation_metrics.items())
    lines.extend(["", "## Augmentations", ""])
    lines.extend(f"- {item}" for item in (augmentations or ["None"]))
    if notes:
        lines.extend(["", "## Notes", ""])
        lines.extend(f"- {item}" for item in notes)
    return "\n".join(lines) + "\n"


def write_technical_report(content: str, path: Path | str) -> Path:
    """Write a UTF-8 technical report, creating its parent directory."""
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding="utf-8")
    return destination


def record_run(
    output_dir: Path | str,
    *,
    task: str,
    model: str,
    configuration: Any,
    validation_metrics: Mapping[str, float],
    split: SplitResult | None = None,
    data_summary: Mapping[str, Any] | None = None,
    inference_options: Mapping[str, Any] | None = None,
    checkpoint: str | None = None,
    notes: list[str] | None = None,
) -> Path:
    """Save configuration/fold positions and a matching human-readable technical report."""
    root = Path(output_dir)
    split_summary = (
        {"train_rows": len(split.train), "valid_rows": len(split.valid)} if split else {}
    )
    payload = {
        "task": task,
        "model": model,
        "configuration": _mapping(configuration),
        "validation_metrics": dict(validation_metrics),
        "data_summary": dict(data_summary or {}),
        "inference_options": dict(inference_options or {}),
        "checkpoint": checkpoint,
        "notes": notes or [],
        "split": {
            **split_summary,
            "train_indices": split.train_indices if split else [],
            "valid_indices": split.valid_indices if split else [],
        },
    }
    save_json(payload, root / "run.json")
    report = technical_report(
        task=task,
        model=model,
        data_summary=data_summary or {},
        split_summary=split_summary,
        training_config=configuration,
        validation_metrics=validation_metrics,
        checkpoint=checkpoint,
        notes=[*(notes or []), f"Inference options: {dict(inference_options or {})}"],
    )
    return write_technical_report(report, root / "technical_report.md")


raise NotImplementedError('TODO: align predictions to sample IDs and validate the complete file')
roundtrip = pd.read_csv(
    destination, converters={ID_COLUMN: str}, keep_default_na=False, float_precision="round_trip"
)
assert roundtrip[ID_COLUMN].tolist() == sample[ID_COLUMN].tolist()
assert roundtrip.columns.tolist() == sample.columns.tolist()
assert len(roundtrip) == len(sample)
pd.testing.assert_frame_equal(roundtrip.astype(str), submission.astype(str), check_dtype=False)
save_json(
    {"train_ids": tr[ID_COLUMN].tolist(), "validation_ids": va[ID_COLUMN].tolist()},
    OUTPUT / "fold_ids.json",
)
save_json(
    {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "seed": SEED,
        "max_rows": MAX_ROWS,
        "image_size": IMAGE_SIZE,
        "model_name": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "embedding_dim": EMBEDDING_DIM,
        "imbalance_weights": USE_IMBALANCE_WEIGHTS,
        "history_features": USE_HISTORY_FEATURES,
        "pretrained_allowed": PRETRAINED_ALLOWED,
        "initial_weight_path": WEIGHT_PATH,
        "asset_provenance": ASSET_PROVENANCE,
        "trainer": globals().get("training"),
    },
    OUTPUT / "lesson_config.json",
)
record_run(
    OUTPUT,
    task=FAMILY,
    model=MODEL_DESCRIPTION,
    configuration=competition.to_dict(),
    validation_metrics={
        "official_score": float(validation_score),
        "fit_seconds": fit_seconds,
        "inference_seconds": inference_seconds,
    },
    split=split,
    inference_options={
        "threshold": float(selected_threshold),
        "label_order": TARGET_COLUMNS,
        "image_size": IMAGE_SIZE,
        "max_length": MAX_LENGTH,
        "model_name": MODEL_NAME,
        "asset_provenance": ASSET_PROVENANCE,
    },
    checkpoint=str(checkpoint_path),
    notes=[
        "Educational candidate, not a reproduced Kaggle leaderboard score.",
        "No test labels, test pseudo-labeling or unauthorized external data used.",
        f"Imbalance weights enabled: {USE_IMBALANCE_WEIGHTS}",
    ],
)
pd.DataFrame(
    [
        {
            "run": RUN_ID,
            "family": FAMILY,
            "variant": VARIANT,
            "score": validation_score,
            "fit_seconds": fit_seconds,
            "inference_seconds": inference_seconds,
        }
    ]
).to_csv(OUTPUT / "comparison_row.csv", index=False)
# Save your edited notebook/source alongside this folder before transferring the bundle.
copied = sync_artifacts(OUTPUT, DURABLE_OUTPUT)
print("Checked local CSV:", destination, "Durable files copied:", len(copied))

# ruff: noqa: E402, F401


<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>@dataclass(slots=True)
class SubmissionValidation:
    &quot;&quot;&quot;Validation errors and non-fatal warnings for a candidate submission.&quot;&quot;&quot;

    valid: bool
    errors: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)

    def raise_for_errors(self) -&gt; None:
        &quot;&quot;&quot;Raise one ``ValueError`` containing every validation error.&quot;&quot;&quot;
        if self.errors:
            raise ValueError(&quot;Invalid submission: &quot; + &quot;; &quot;.join(self.errors))


def build_submission(
    sample: pd.DataFrame,
    predictions: Mapping[str, Sequence[object]],
    *,
    prediction_ids: pd.DataFrame | Sequence[object] | None = None,
    id_columns: str | Iterable[str] | None = None,
) -&gt; pd.DataFrame:
    &quot;&quot;&quot;Copy the sample and optionally align predictions using unique (possibly composite) IDs.

    Without IDs, predictions must already be in sample order. Prefer explicit IDs for inference.
    Alignment rejects missing, duplicate, extra, or absent keys rather than guessing row order.
    &quot;&quot;&quot;
    submission = sample.copy()
    ids = [id_columns] if isinstance(id_columns, str) else list(id_columns or [])
    order = None
    if prediction_ids is not None:
        if not ids:
            raise ValueError(&quot;id_columns is required with prediction_ids&quot;)
        if isinstance(prediction_ids, pd.DataFrame):
            keys = prediction_ids[ids].reset_index(drop=True)
        elif len(ids) == 1:
            keys = pd.DataFrame({ids[0]: list(prediction_ids)})
        else:
            raise ValueError(&quot;Composite prediction IDs must be a DataFrame&quot;)
        for name, frame in ((&quot;sample&quot;, sample), (&quot;predictions&quot;, keys)):
            if (
                frame[ids].isna().any().any()
                or frame[ids].eq(&quot;&quot;).any().any()
                or frame.duplicated(ids).any()
            ):
                raise ValueError(f&quot;{name} IDs must be nonmissing and unique across {ids}&quot;)
        expected = pd.MultiIndex.from_frame(sample[ids])
        actual = pd.MultiIndex.from_frame(keys)
        order = actual.get_indexer(expected)
        if len(actual) != len(expected) or (order &lt; 0).any():
            raise ValueError(&quot;Prediction IDs must match the sample IDs exactly&quot;)
    for column, values in predictions.items():
        if column not in submission.columns:
            raise KeyError(f&quot;Prediction column not present in sample submission: {column}&quot;)
        if len(values) != len(submission):
            raise ValueError(
                f&quot;Column {column} has {len(values)} values; expected {len(submission)}&quot;
            )
        if column in ids:
            raise ValueError(f&quot;Cannot overwrite ID column: {column}&quot;)
        values = list(values)
        submission[column] = values if order is None else [values[index] for index in order]
    return submission


def validate_submission(
    submission: pd.DataFrame,
    sample: pd.DataFrame,
    *,
    id_columns: str | Iterable[str] | None = None,
    allowed_labels: Mapping[str, Iterable[object]] | None = None,
) -&gt; SubmissionValidation:
    &quot;&quot;&quot;Check schema, row count, ID order, missing values, and optional allowed labels.&quot;&quot;&quot;
    errors: list[str] = []
    warnings: list[str] = []
    if submission.columns.duplicated().any() or sample.columns.duplicated().any():
        errors.append(&quot;Submission and sample column names must be unique&quot;)
        return SubmissionValidation(valid=False, errors=errors)
    if list(submission.columns) != list(sample.columns):
        errors.append(
            f&quot;Columns/order differ: got {list(submission.columns)}, expected {list(sample.columns)}&quot;
        )
    if len(submission) != len(sample):
        errors.append(f&quot;Row count differs: got {len(submission)}, expected {len(sample)}&quot;)
    if submission.isna().any().any():
        columns = submission.columns[submission.isna().any()].tolist()
        errors.append(f&quot;Missing values in columns: {columns}&quot;)
    numeric = submission.select_dtypes(include=&quot;number&quot;)
    if not np.isfinite(numeric.to_numpy(dtype=float, na_value=np.nan)).all():
        errors.append(&quot;Non-finite numeric values are not allowed&quot;)
    ids = [id_columns] if isinstance(id_columns, str) else list(id_columns or [])
    for column in ids:
        if column not in submission or column not in sample:
            errors.append(f&quot;Missing ID column: {column}&quot;)
        elif submission[column].isna().any() or submission[column].eq(&quot;&quot;).any():
            errors.append(f&quot;Missing ID values in column: {column}&quot;)
        elif len(submission) == len(sample) and not submission[column].equals(sample[column]):
            errors.append(f&quot;ID values/order changed in column: {column}&quot;)
    if ids and all(column in submission for column in ids) and submission.duplicated(ids).any():
        warnings.append(f&quot;Duplicate ID keys across columns: {ids}&quot;)
    for column, labels in (allowed_labels or {}).items():
        if column not in submission:
            errors.append(f&quot;Missing allowed-label column: {column}&quot;)
        else:
            invalid = set(submission[column].dropna().unique()) - set(labels)
            if invalid:
                errors.append(f&quot;Unexpected labels in {column}: {sorted(map(str, invalid))[:10]}&quot;)
    return SubmissionValidation(valid=not errors, errors=errors, warnings=warnings)


def write_submission(
    submission: pd.DataFrame,
    path: Path | str,
    *,
    sample: pd.DataFrame | None = None,
    id_columns: str | Iterable[str] | None = None,
    allowed_labels: Mapping[str, Iterable[object]] | None = None,
) -&gt; Path:
    &quot;&quot;&quot;Validate then atomically write a UTF-8 CSV submission.&quot;&quot;&quot;
    if sample is not None:
        validation = validate_submission(
            submission, sample, id_columns=id_columns, allowed_labels=allowed_labels
        )
        validation.raise_for_errors()
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f&quot;.{destination.name}.{uuid4().hex}.tmp&quot;)
    try:
        submission.to_csv(temporary, index=False, encoding=&quot;utf-8&quot;)
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination


def _mapping(value: Any) -&gt; Mapping[str, Any]:
    if is_dataclass(value):
        return asdict(value)
    if isinstance(value, Mapping):
        return value
    raise TypeError(&quot;Expected a dataclass or mapping&quot;)


def technical_report(
    *,
    task: str,
    model: str,
    data_summary: Mapping[str, Any],
    split_summary: Mapping[str, Any],
    training_config: Any,
    validation_metrics: Mapping[str, float],
    augmentations: list[str] | None = None,
    checkpoint: str | None = None,
    notes: list[str] | None = None,
) -&gt; str:
    &quot;&quot;&quot;Render the standard competition handoff report as Markdown text.&quot;&quot;&quot;
    config = _mapping(training_config)
    lines = [
        &quot;# Technical Report&quot;,
        &quot;&quot;,
        f&quot;- Task: {task}&quot;,
        f&quot;- Model: {model}&quot;,
        f&quot;- Checkpoint: {checkpoint or &#x27;not recorded&#x27;}&quot;,
        &quot;&quot;,
        &quot;## Data and validation&quot;,
        &quot;&quot;,
    ]
    lines.extend(f&quot;- {key}: {value}&quot; for key, value in data_summary.items())
    lines.extend(f&quot;- Split {key}: {value}&quot; for key, value in split_summary.items())
    lines.extend([&quot;&quot;, &quot;## Training&quot;, &quot;&quot;])
    lines.extend(f&quot;- {key}: {value}&quot; for key, value in config.items())
    lines.extend([&quot;&quot;, &quot;## Validation metrics&quot;, &quot;&quot;])
    lines.extend(f&quot;- {key}: {value:.6f}&quot; for key, value in validation_metrics.items())
    lines.extend([&quot;&quot;, &quot;## Augmentations&quot;, &quot;&quot;])
    lines.extend(f&quot;- {item}&quot; for item in (augmentations or [&quot;None&quot;]))
    if notes:
        lines.extend([&quot;&quot;, &quot;## Notes&quot;, &quot;&quot;])
        lines.extend(f&quot;- {item}&quot; for item in notes)
    return &quot;\n&quot;.join(lines) + &quot;\n&quot;


def write_technical_report(content: str, path: Path | str) -&gt; Path:
    &quot;&quot;&quot;Write a UTF-8 technical report, creating its parent directory.&quot;&quot;&quot;
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding=&quot;utf-8&quot;)
    return destination


def record_run(
    output_dir: Path | str,
    *,
    task: str,
    model: str,
    configuration: Any,
    validation_metrics: Mapping[str, float],
    split: SplitResult | None = None,
    data_summary: Mapping[str, Any] | None = None,
    inference_options: Mapping[str, Any] | None = None,
    checkpoint: str | None = None,
    notes: list[str] | None = None,
) -&gt; Path:
    &quot;&quot;&quot;Save configuration/fold positions and a matching human-readable technical report.&quot;&quot;&quot;
    root = Path(output_dir)
    split_summary = (
        {&quot;train_rows&quot;: len(split.train), &quot;valid_rows&quot;: len(split.valid)} if split else {}
    )
    payload = {
        &quot;task&quot;: task,
        &quot;model&quot;: model,
        &quot;configuration&quot;: _mapping(configuration),
        &quot;validation_metrics&quot;: dict(validation_metrics),
        &quot;data_summary&quot;: dict(data_summary or {}),
        &quot;inference_options&quot;: dict(inference_options or {}),
        &quot;checkpoint&quot;: checkpoint,
        &quot;notes&quot;: notes or [],
        &quot;split&quot;: {
            **split_summary,
            &quot;train_indices&quot;: split.train_indices if split else [],
            &quot;valid_indices&quot;: split.valid_indices if split else [],
        },
    }
    save_json(payload, root / &quot;run.json&quot;)
    report = technical_report(
        task=task,
        model=model,
        data_summary=data_summary or {},
        split_summary=split_summary,
        training_config=configuration,
        validation_metrics=validation_metrics,
        checkpoint=checkpoint,
        notes=[*(notes or []), f&quot;Inference options: {dict(inference_options or {})}&quot;],
    )
    return write_technical_report(report, root / &quot;technical_report.md&quot;)


submission = build_submission(
    sample, output_values, prediction_ids=test[ID_COLUMN], id_columns=ID_COLUMN
)
validate_submission(submission, sample, id_columns=ID_COLUMN).raise_for_errors()
destination = write_submission(submission, OUTPUT / &quot;submission.csv&quot;)
roundtrip = pd.read_csv(
    destination, converters={ID_COLUMN: str}, keep_default_na=False, float_precision=&quot;round_trip&quot;
)
assert roundtrip[ID_COLUMN].tolist() == sample[ID_COLUMN].tolist()
assert roundtrip.columns.tolist() == sample.columns.tolist()
assert len(roundtrip) == len(sample)
pd.testing.assert_frame_equal(roundtrip.astype(str), submission.astype(str), check_dtype=False)
save_json(
    {&quot;train_ids&quot;: tr[ID_COLUMN].tolist(), &quot;validation_ids&quot;: va[ID_COLUMN].tolist()},
    OUTPUT / &quot;fold_ids.json&quot;,
)
save_json(
    {
        &quot;epochs&quot;: EPOCHS,
        &quot;batch_size&quot;: BATCH_SIZE,
        &quot;seed&quot;: SEED,
        &quot;max_rows&quot;: MAX_ROWS,
        &quot;image_size&quot;: IMAGE_SIZE,
        &quot;model_name&quot;: MODEL_NAME,
        &quot;max_length&quot;: MAX_LENGTH,
        &quot;embedding_dim&quot;: EMBEDDING_DIM,
        &quot;imbalance_weights&quot;: USE_IMBALANCE_WEIGHTS,
        &quot;history_features&quot;: USE_HISTORY_FEATURES,
        &quot;pretrained_allowed&quot;: PRETRAINED_ALLOWED,
        &quot;initial_weight_path&quot;: WEIGHT_PATH,
        &quot;asset_provenance&quot;: ASSET_PROVENANCE,
        &quot;trainer&quot;: globals().get(&quot;training&quot;),
    },
    OUTPUT / &quot;lesson_config.json&quot;,
)
record_run(
    OUTPUT,
    task=FAMILY,
    model=MODEL_DESCRIPTION,
    configuration=competition.to_dict(),
    validation_metrics={
        &quot;official_score&quot;: float(validation_score),
        &quot;fit_seconds&quot;: fit_seconds,
        &quot;inference_seconds&quot;: inference_seconds,
    },
    split=split,
    inference_options={
        &quot;threshold&quot;: float(selected_threshold),
        &quot;label_order&quot;: TARGET_COLUMNS,
        &quot;image_size&quot;: IMAGE_SIZE,
        &quot;max_length&quot;: MAX_LENGTH,
        &quot;model_name&quot;: MODEL_NAME,
        &quot;asset_provenance&quot;: ASSET_PROVENANCE,
    },
    checkpoint=str(checkpoint_path),
    notes=[
        &quot;Educational candidate, not a reproduced Kaggle leaderboard score.&quot;,
        &quot;No test labels, test pseudo-labeling or unauthorized external data used.&quot;,
        f&quot;Imbalance weights enabled: {USE_IMBALANCE_WEIGHTS}&quot;,
    ],
)
pd.DataFrame(
    [
        {
            &quot;run&quot;: RUN_ID,
            &quot;family&quot;: FAMILY,
            &quot;variant&quot;: VARIANT,
            &quot;score&quot;: validation_score,
            &quot;fit_seconds&quot;: fit_seconds,
            &quot;inference_seconds&quot;: inference_seconds,
        }
    ]
).to_csv(OUTPUT / &quot;comparison_row.csv&quot;, index=False)
# Save your edited notebook/source alongside this folder before transferring the bundle.
copied = sync_artifacts(OUTPUT, DURABLE_OUTPUT)
print(&quot;Checked local CSV:&quot;, destination, &quot;Durable files copied:&quot;, len(copied))

# ruff: noqa: E402, F401
</code></pre>
</details>
